In [1]:
import os
import cv2

In [6]:
import image_builder
import rcprinter_client

Simulate reading img from zulip server

In [3]:
img_from_server = open('../data/image04.jpeg', 'rb').read()

Run through parser

In [5]:
transformed_image_as_bytes = image_builder.parse_form(img_from_server)

Send to printer

In [7]:
c = rcprinter_client.RCPrinterClient()

In [8]:
c.send_image(transformed_image_as_bytes)

200 - b'{}'


# SQLITE

In [1]:
import datetime
import sqlite3

In [2]:
conn = sqlite3.connect(":memory:")

In [3]:
def now_timestamp_s():
    return int(datetime.datetime.now().timestamp())

In [4]:
def create_game(conn, length, created_by) -> int:
    res = conn.execute(
        """
        INSERT INTO game(length, createdBy, createdAt) VALUES(?, ?, ?);
        """,
        (length, created_by, now_timestamp_s()),
    )
    return res.lastrowid

In [5]:
conn.execute("""
    CREATE TABLE IF NOT EXISTS game(
        id INTEGER PRIMARY KEY,
        length INTEGER NOT NULL,
        createdBy INTEGER NOT NULL,
        createdAt INTEGER NOT NULL
    );
""")
conn.execute("""
    CREATE TABLE IF NOT EXISTS drawing(
        id INTEGER PRIMARY KEY,
        gameId INTEGER NOT NULL,
        drawingNumber INTEGER NOT NULL,
        artistZulipId INTEGER NOT NULL,
        imageData BLOB,
        createdAt INTEGER NOT NULL,
        submittedAt INTEGER
    );
""")

In [10]:
def get_next_drawing_number(conn, game_id):
    res = conn.execute(
        """
        SELECT
            MAX(drawingNumber)
        FROM
            drawing
        WHERE
            gameId = ?;
        """,
        (game_id, ),
    )

    last_number = res.fetchone()[0]

    if last_number is None:
        return 1
    
    return last_number + 1

def init_drawing(
    conn,
    game_id,
    artist_zulip_id,
):
    next_drawing_number = get_next_drawing_number(conn, game_id)
    
    res = conn.execute(
        """
        INSERT INTO drawing(gameId, drawingNumber, artistZulipId, createdAt) VALUES(?, ?, ?, ?);
        """,
        (game_id, next_drawing_number, artist_zulip_id, now_timestamp_s()),
    )

    return res.lastrowid

In [7]:
game_id = create_game(conn, 10, 192)

In [11]:
drawing_id = init_drawing(conn, game_id, 192)

In [ ]:
    CREATE TABLE IF NOT EXISTS drawing(
        id INTEGER PRIMARY KEY,
        gameId INTEGER NOT NULL,
        drawingNumber INTEGER NOT NULL,
        artistZulipId INTEGER NOT NULL,
        imageData BLOB,
        createdAt INTEGER NOT NULL,
        submittedAt INTEGER
    );

In [12]:
def submit_drawing(
    conn,
    drawing_id,
    image_data,
):
    conn.execute(
        """
        UPDATE
            drawing
        SET
            imageData = ?,
            submittedAt = ?
        WHERE
            id = ?;
        """,
        (image_data, now_timestamp_s(), drawing_id),
    )

In [13]:
def get_drawing(conn, drawing_id):
    res = conn.execute(
        """
        SELECT *
        FROM drawing
        WHERE id = ?;
        """,
        (drawing_id, ),
    )
    return res.fetchone()

In [14]:
get_drawing(conn, drawing_id)

(2, 1, 2, 192, None, 1779997770, None)

In [17]:
image_data = open('../data/image00.jpg', 'rb').read()

In [18]:
submit_drawing(
    conn,
    drawing_id,
    image_data,
)

In [ ]:
get_drawing(conn, drawing_id)

In [ ]:
import datetime
import sqlite3

conn = sqlite3.connect(":memory:")

def now_timestamp_s():
    return int(datetime.datetime.now().timestamp())

def create_game(conn, length, created_by) -> int:
    res = conn.execute(
        """
        INSERT INTO game(length, createdBy, createdAt) VALUES(?, ?, ?);
        """,
        (length, created_by, now_timestamp_s()),
    )
    return res.lastrowid

conn.execute("""
    CREATE TABLE IF NOT EXISTS game(
        id INTEGER PRIMARY KEY,
        length INTEGER NOT NULL,
        createdBy INTEGER NOT NULL,
        createdAt INTEGER NOT NULL
    );
""")
conn.execute("""
    CREATE TABLE IF NOT EXISTS drawing(
        id INTEGER PRIMARY KEY,
        gameId INTEGER NOT NULL,
        drawingNumber INTEGER NOT NULL,
        artistZulipId INTEGER NOT NULL,
        imageData BLOB,
        createdAt INTEGER NOT NULL,
        submittedAt INTEGER
    );
""")

def get_next_drawing_number(conn, game_id):
    res = conn.execute(
        """
        SELECT
            MAX(drawingNumber)
        FROM
            drawing
        WHERE
            gameId = ?;
        """,
        (game_id, ),
    )

    last_number = res.fetchone()[0]

    if last_number is None:
        return 1
    
    return last_number + 1

def init_drawing(
    conn,
    game_id,
    artist_zulip_id,
):
    next_drawing_number = get_next_drawing_number(conn, game_id)
    
    res = conn.execute(
        """
        INSERT INTO drawing(gameId, drawingNumber, artistZulipId, createdAt) VALUES(?, ?, ?, ?);
        """,
        (game_id, next_drawing_number, artist_zulip_id, now_timestamp_s()),
    )

    return res.lastrowid

game_id = create_game(conn, 10, 192)

drawing_id = init_drawing(conn, game_id, 192)

    CREATE TABLE IF NOT EXISTS drawing(
        id INTEGER PRIMARY KEY,
        gameId INTEGER NOT NULL,
        drawingNumber INTEGER NOT NULL,
        artistZulipId INTEGER NOT NULL,
        imageData BLOB,
        createdAt INTEGER NOT NULL,
        submittedAt INTEGER
    );

def submit_drawing(
    conn,
    drawing_id,
    image_data,
):
    conn.execute(
        """
        UPDATE
            drawing
        SET
            imageData = ?,
            submittedAt = ?
        WHERE
            id = ?;
        """,
        (image_data, now_timestamp_s(), drawing_id),
    )

def get_drawing(conn, drawing_id):
    res = conn.execute(
        """
        SELECT *
        FROM drawing
        WHERE id = ?;
        """,
        (drawing_id, ),
    )
    return res.fetchone()

get_drawing(conn, drawing_id)

image_data = open('../data/image00.jpg', 'rb').read()

submit_drawing(
    conn,
    drawing_id,
    image_data,
)

get_drawing(conn, drawing_id)